<a href="https://colab.research.google.com/github/zyf-hitsz/PytorchLearning/blob/main/%E7%A5%9E%E7%BB%8F%E7%BD%91%E7%BB%9C/%E5%8D%B7%E7%A7%AF%E5%B1%82/ConvolutionLayers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#神经网络--卷积层
---
我们学习一维卷积、二维卷积、三位卷积层的定义方法和参数意义。此外我们还学习一下转置卷积。

一维卷积只沿着序列方向这一个方向移动计算，通常用处处理时间序列、语音、通信信号，进行信号处理、特征序列提取等操作。

二维卷积沿着两个方向滑动，最常用于图像处理，进行图像分类、检测、分割等操作。

三维卷积沿着三个方向滑动，常用于视频、CT/MRI、三维空间数据，进行视频动作识别、医学影像处理等操作。

一维、二维、三维卷积其实并没有什么显著的不同点，只是要注意数据维度本身的意义和不同的数据维度带来的参数维度。只要根据我们要学习的数据维度选择卷积维度，然后根据目的和数据特点设置合理的参数。






##先看一维卷积

In [ ]:
#定义一个卷积层
class torch.nn.Conv1d(in_channels, out_channels, kernel_size, stride=1,
          padding=0, dilation=1, groups=1, bias=True, padding_mode='zeros', device=None, dtype=None)
#in_channels: 输入数据的通道数，也即维数。
#out_channels: 卷积层产生的输出通道数。这决定了卷积核的数量，每个卷积核都会生成一个特征图。
#kernel_size: 卷积核（或称为滤波器）的大小。对于一维卷积，这表示卷积核的宽度。
#stride (默认为 1): 卷积核的步长。它决定了卷积核在输入数据上移动的距离。
#padding (默认为 0): 填充。在输入数据的两端填充的数据的数量。填充通常用于保持输出特征图的尺寸或防止信息丢失。
  #如果 padding=1，则会在输入序列的两端各添加一个零。
#dilation (默认为 1): 扩张率。它决定了卷积核中各元素之间的间距。
  #如 dilation=2，卷积核的元素之间会有1个跳过的像素，这样可以扩大感受野而不需要增加参数数量。
#groups (默认为 1): 分组卷积。如果 groups > 1，输入和输出通道将被分组，并且每个组独立地进行卷积。
  #这在某些网络结构中用于减少计算量。
#bias (默认为 True): 如果为 True，则卷积层会学习一个偏置向量并加到输出中。
#padding_mode (默认为 'zeros'): 填充的类型。除了 'zeros'（零填充），还可以是 'reflect' 或 'replicate'（反射或复制填充）。
#device (默认为 None): 指定张量所在的设备，例如 'cuda' 或 'cpu'。
#dtype (默认为 None): 指定张量的数据类型，例如 torch.float32。

##这里我们重点看一下padding_mode.

padding_mode 决定了在进行卷积操作之前，如何填充输入张量的边界。

Pytorch中的Conv1d主要支持一下几种padding_mode：

1、zeros（默认）：用常数0填充边界，假设序列之外的数据都是0.

实例：输入[1,2,3],padding_mode='zeros',结果为[0,...,0,1,2,3,0,...,0]

2、reflect（反射填充）：一边缘为轴，将输入数据就镜像反射。

实例：输入[1,2,3],输出[3,2,1,2,3,2,1]。

3、replicate（复制填充）：直接复制输入张量的边缘值进行填充。

实例：输入[1,2,3]，输出[1,1,2,3,3,]

4、circular（循环填充）：将序列看作是周期性的进行填充。

实例：输入[1,2,3],输出[1,2,3,1,2,3,1,2,3,]

---

### Padding 后的长度计算

输入在经过 padding 后的长度由以下因素确定：
1. **原始输入长度 ($L_{in}$)**
2. **`padding` 参数值**：指在序列两端各添加的元素数量。

**计算公式：**
$$L_{padded} = L_{in} + 2 \times padding$$

**示例：**
如果输入长度 $L_{in} = 3$，设置 `padding = 1`，则：
$$L_{padded} = 3 + 2 \times 1 = 5$$
填充后的序列长度变为 5。

### 分组卷积 (Groups) 约束条件

`groups` 参数必须满足以下要求：
1. **范围**: $1 \le groups \le in\_channels$。
2. **整除性**: `in_channels` 和 `out_channels` 都必须能被 `groups` 整除。

- **groups=1**: 标准卷积。
- **groups=in_channels**: 深度卷积 (Depthwise Convolution)。

**参数量计算公式：**
$$\text{Parameters} = \text{out_channels} \times \frac{\text{in_channels}}{\text{groups}} \times \text{kernel_size}$$

In [ ]:
import torch
import torch.nn as nn

input_data = torch.randn(1, 4, 10)
groups_num=4
conv = nn.Conv1d(in_channels=4, out_channels=8, kernel_size=3, groups=groups_num)
output = conv(input_data)

print(f"输入形状: {input_data.shape}")
print(f"groups={groups_num}")
print(f"输出形状: {output.shape}")
print(f"卷积核权重形状 (out_channels, in_channels/groups, kernel_size):{conv.weight.shape}")


输入形状: torch.Size([1, 4, 10])
groups=4
输出形状: torch.Size([1, 8, 8])
卷积核权重形状 (out_channels, in_channels/groups, kernel_size):torch.Size([8, 1, 3])


##再看二维卷积


In [ ]:
import torch


# 定义一个二维卷积层
class torch.nn.Conv2d(in_channels, out_channels, kernel_size, stride=1,
          padding=0, dilation=1, groups=1, bias=True, padding_mode='zeros', device=None, dtype=None)

# in_channels: 输入图像的通道数。例如 RGB 图像为 3，黑白图像为 1。
# out_channels: 卷积产生的输出通道数（即卷积核的数量）。
# kernel_size: 卷积核的大小。可以是一个整数（如 3 代表 3x3），也可以是一个元组 (kH, kW)。
# stride: 卷积的步长。决定了卷积核在水平和垂直方向移动的像素数。默认 1。
# padding: 填充。在输入的四周添加数据。默认 0。
# dilation: 扩张率。控制卷积核内部各点之间的距离。
# groups: 分组卷积。控制输入通道和输出通道之间的连接。in_channels 和 out_channels 必须都能被它整除。
# bias: 是否添加可学习的偏置。默认 True。
# padding_mode: 填充模式，如 'zeros', 'reflect', 'replicate' 或 'circular'。

一个最简单的例子

In [ ]:
# With square kernels and equal stride
m = nn.Conv2d(16, 33, 3, stride=2)
# non-square kernels and unequal stride and with padding
m = nn.Conv2d(16, 33, (3, 5), stride=(2, 1), padding=(4, 2))
# non-square kernels and unequal stride and with padding and dilation
m = nn.Conv2d(16, 33, (3, 5), stride=(2, 1), padding=(4, 2), dilation=(3, 1))
input = torch.randn(20, 16, 50, 100)
output = m(input)

##再看三维卷积

In [ ]:
import torch
import torch.nn as nn

# 定义一个三维卷积层
class torch.nn.Conv3d(in_channels, out_channels, kernel_size, stride=1,
          padding=0, dilation=1, groups=1, bias=True, padding_mode='zeros', device=None, dtype=None)

# --- 三维卷积的特殊点 ---
# 1. 维度：它在 (D, H, W) 三个空间维度上滑动，而不仅仅是 (H, W)。
# 2. 输入形状：期望 5D 输入张量 (N, C, D, H, W)。
#    - N: Batch Size
#    - C: 通道数 (Channels)
#    - D: 深度 (Depth/Time)，例如视频的帧数或 CT 的切片数
#    - H/W: 每一帧/切片的高度和宽度

# --- 参数含义 ---
# in_channels: 输入的通道数。
# out_channels: 输出的通道数（卷积核数量）。
# kernel_size: 卷积核大小。可以是一个值 (D=H=W) 或元组 (kD, kH, kW)。
# stride: 步长。在三个维度上的滑动间隔，可为元组 (sD, sH, sW)。
# padding: 填充。在 (D, H, W) 各维度的两端填充，可为元组 (pD, pH, pW)。
# dilation: 扩张率。控制卷积核元素间的间距。
# groups: 分组卷积。用于控制输入与输出通道的连接关系。
# bias: 是否使用偏置。

---
##转置卷积
ConvTraapose1d（一维转置卷积），ConvTraapose2d（二维转置卷积），ConvTraapose3d（三维转置卷积）。转置卷积和普通卷积的最大区别在于：普通卷积主要用于特征提取和下采样；转置卷积主要用于特征恢复和上采样。即普通卷积从大尺度数据（图像、视频、序列）中提取小尺度特征，而转置卷积则是根据小尺度，恢复生产大尺度特征，如输出音视频、图像等。

卷积可以写成矩阵乘法
$$y=Wx,$$
其中：x：输入,W：卷积对应的大矩阵。

那么转置卷积就可以对应为：
$$x'=W^Ty，$$
也就是卷积矩阵的对应操作。

具体到卷积操作，

ConvNd:在N个反向滑动窗口提取局部特征；

ConvTransposeNd:在N个方向扩展特征空间并学习映射。
---
具体地，我们看一下二维转置卷积：





### 二维转置卷积

```python
class torch.nn.ConvTranspose2d(in_channels, out_channels, kernel_size, stride=1,
                padding=0, output_padding=0, groups=1, bias=True, dilation=1, padding_mode='zeros', device=None, dtype=None)
```

#### 1. 参数意义
*   **in_channels**: 输入特征图的通道数。
*   **out_channels**: 经过转置卷积后输出的通道数。
*   **kernel_size**: 卷积核大小。与普通卷积一致，决定了“扩散”的范围。
*   **stride**: **核心不同点**。在普通卷积中，stride > 1 会减小尺寸；在转置卷积中，**stride > 1 会成倍放大输出尺寸**（上采样）。
*   **padding**: **核心不同点**。在普通卷积中，padding 增加输入尺寸；在转置卷积中，**padding 反而会减小输出尺寸**（它是从逻辑上的输入中“裁剪”掉的部分）。
*   **output_padding (特有参数)**: 用于补充由于 stride > 1 时造成的输出尺寸歧义。它只在输出的一侧填充，确保输出形状符合预期。
*   **dilation/groups/bias**: 与普通卷积含义相同。

#### 2. 转置卷积与普通卷积的区别总结
| 特性 | 普通卷积 (Conv2d) | 转置卷积 (ConvTranspose2d) |
| :--- | :--- | :--- |
| **主要用途** | 特征提取、下采样（缩小尺寸） | 内容生成、上采样（放大尺寸） |
| **Stride** | 步长越大，输出越小 | 步长越大，输出越大 |
| **Padding** | 填充越多，输出越大 | 填充越多，输出越小 |
| **计算逻辑** | 多对一（核覆盖区域求和） | 一对多（输入像素乘核扩散） |

In [2]:
import torch
import torch.nn as nn

# 假设输入是一个 2x2 的特征图
input_tensor = torch.randn(1, 16, 2, 2)

# 定义转置卷积：设置 stride=2 将尺寸翻倍
# 输出尺寸计算公式：H_out = (H_in - 1) * stride - 2 * padding + dilation * (kernel_size - 1) + output_padding + 1
transpose_conv = nn.ConvTranspose2d(in_channels=16, out_channels=8, kernel_size=3, stride=2, padding=1, output_padding=1)

output = transpose_conv(input_tensor)

print(f"输入形状: {input_tensor.shape}")
print(f"输出形状:{output.shape}")

输入形状: torch.Size([1, 16, 2, 2])
输出形状:torch.Size([1, 8, 4, 4])
